#Assignment 9

In [ ]:
%pip install opencv-python easyocr deepface pyttsx3 tf-keras

##Question 1

In [ ]:
import cv2
import easyocr
import re
import difflib
import pyttsx3


KNOWN_DRUGS = (
    "paracetamol", "amoxicillin", "ibuprofen", "metformin",
    "cetirizine", "atorvastatin", "aspirin", "omeprazole"
)

DOSAGE_PATTERN = re.compile(r"\b\d+\s*(?:mg|ml|mcg|g)\b", re.IGNORECASE)
FREQUENCY_PATTERN = re.compile(
    r"\b(?:once|twice|thrice|\d+\s*times)\s*(?:a|per)?\s*day\b",
    re.IGNORECASE
)
EXPIRY_PATTERN = re.compile(r"\b(0[1-9]|1[0-2])[/-](20\d{2}|\d{2})\b")


def read_label(image_path):
    """Read visible label text from a medicine-bottle image."""
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(
            f"Could not open {image_path!r}. Check that the image exists."
        )

    reader = easyocr.Reader(["en"], gpu=False)
    detected_lines = reader.readtext(image, detail=0)
    return " ".join(detected_lines)


def identify_medicine(text):
    """Find the closest known medicine name among the detected words."""
    words = re.findall(r"\b[A-Za-z]{4,}\b", text.lower())
    for word in words:
        candidates = difflib.get_close_matches(
            word, KNOWN_DRUGS, n=1, cutoff=0.7
        )
        if candidates:
            return candidates[0].capitalize()
    return "Unknown medication"


def extract_first(pattern, text, fallback):
    """Return the first regex match, or a readable fallback."""
    found = pattern.search(text)
    return found.group(0) if found else fallback


def main():
    raw_text = read_label("medicine_bottle.jpg")

    medicine = identify_medicine(raw_text)
    dosage = extract_first(DOSAGE_PATTERN, raw_text, "Not specified")
    frequency = extract_first(FREQUENCY_PATTERN, raw_text, "Not specified")
    expiry = extract_first(EXPIRY_PATTERN, raw_text, "Not found")

    spoken_summary = (
        f"Medicine spotted: {medicine}. "
        f"Label dosage: {dosage}. "
        f"Daily schedule: {frequency}. "
        f"Expiry date: {expiry}."
    )

    speaker = pyttsx3.init()
    speaker.say(spoken_summary)
    speaker.runAndWait()


if __name__ == "__main__":
    main()


##Question 2

In [ ]:
import cv2
from deepface import DeepFace


MOOD_MESSAGES = {
    "happy": "Sunshine alert! Your smile is doing overtime!",
    "sad": "Cloudy vibes detected. Make it Rain!",
    "angry": "Steam engine mode! Pause, breathe, and Fire.",
    "neutral": "Chill mode activated. Cool as a cucumber.",
    "surprise": "Plot twist face! What just happened?!",
    "fear": "Brave mode loading... one slow breath at a time!",
    "disgust": "Big nope energy detected. Let's change the vibe!",
}


def analyze_expression(frame):
    """Return the dominant emotion and its face region."""
    analysis = DeepFace.analyze(
        img=frame,
        actions=["emotion"],
        enforce_detection=False
    )

    details = analysis[0] if isinstance(analysis, list) else analysis
    emotion = details.get("dominant_emotion", "neutral")
    region = details.get("region", {})
    face_box = (
        region.get("x", 0), region.get("y", 0),
        region.get("w", 0), region.get("h", 0)
    )
    return emotion, face_box


def draw_status(frame, emotion, message, face_box):
    """Draw the face rectangle and mood caption on the current frame."""
    x, y, width, height = face_box
    if width > 0 and height > 0:
        cv2.rectangle(
            frame, (x, y), (x + width, y + height), (0, 255, 0), 2
        )

    cv2.putText(
        frame, f"Mood: {emotion}", (25, 35),
        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 255, 0), 2
    )
    cv2.putText(
        frame, message[:75], (25, 70),
        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2
    )
    return frame


def main():
    camera = cv2.VideoCapture(0)
    if not camera.isOpened():
        print("Camera couldn't start. Check your webcam connection.")
        return

    frame_number = 0
    current_mood = "neutral"
    current_message = MOOD_MESSAGES[current_mood]
    face_box = (0, 0, 0, 0)

    try:
        while True:
            success, frame = camera.read()
            if not success:
                print("No frame received from the camera.")
                break

            frame_number += 1


            if frame_number % 20 == 0:
                try:
                    current_mood, face_box = analyze_expression(frame)
                    current_message = MOOD_MESSAGES.get(
                        current_mood,
                        "Mystery mood unlocked. Keep being you!"
                    )
                except Exception:

                    current_message = "Mood radar taking a tiny coffee break..."

            preview = draw_status(
                frame, current_mood, current_message, face_box
            )
            cv2.imshow("Mood Buddy", preview)

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
